# Notebook 1: PCB Dataset Audit, 10% Split & Sanity Check

**Mục tiêu**
- Kiểm tra toàn bộ dataset: số ảnh, số nhãn, thống kê bbox
- Tạo split labeled 10% có thể tái lập cho semi-supervised training
- Sanity check: parse bbox, phân bố class, thống kê object nhỏ
- Visualize ảnh đại diện
- Lưu artifact: split JSON, stats JSON, sample images

In [ ]:
import os
import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from sklearn.model_selection import StratifiedShuffleSplit
import yaml

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Paths ─────────────────────────────────────────────────────
DATA_ROOT = Path("/kaggle/input/datasets/chungkein/datapcb-v1/YOLO_format")

TRAIN_IMG = DATA_ROOT / "train/images"
TRAIN_LBL = DATA_ROOT / "train/labels"

VALID_IMG = DATA_ROOT / "valid/images"
VALID_LBL = DATA_ROOT / "valid/labels"

TEST_IMG = DATA_ROOT / "test/images"
TEST_LBL = DATA_ROOT / "test/labels"

YAML_FILE = DATA_ROOT / "data.yaml"

OUT_DIR = Path("/kaggle/working/artifacts/nb01")
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 640
LABELED_RATIO = 0.1

# ── Load dataset metadata from YAML ───────────────────────────
with open(YAML_FILE, "r") as f:
    data_cfg = yaml.safe_load(f)

# Use names from YAML if available
CLASS_NAMES = [
    str(x) for x in data_cfg.get(
        "names",
        list(range(int(data_cfg.get("nc", 0))))
    )
]

NC = int(data_cfg.get("nc", len(CLASS_NAMES)))

# ── Verify paths ──────────────────────────────────────────────
for p in [
    TRAIN_IMG, TRAIN_LBL,
    VALID_IMG, VALID_LBL,
    TEST_IMG, TEST_LBL
]:
    assert p.exists(), f"Path not found: {p}"

print(f"YAML_FILE    : {YAML_FILE}")
print(f"TRAIN_IMG    : {TRAIN_IMG}")
print(f"TRAIN_LBL    : {TRAIN_LBL}")
print(f"VALID_IMG    : {VALID_IMG}")
print(f"VALID_LBL    : {VALID_LBL}")
print(f"TEST_IMG     : {TEST_IMG}")
print(f"TEST_LBL     : {TEST_LBL}")
print(f"OUT_DIR      : {OUT_DIR}")
print(f"SEED         : {SEED}")
print(f"LABELED_RATIO: {LABELED_RATIO:.0%}")
print(f"CLASS_NAMES  : {CLASS_NAMES}")
print(f"NC           : {NC}")

## 1. Dataset Inventory

In [ ]:
def scan_split(img_dir: Path, lbl_dir: Path) -> pd.DataFrame:
    """Scan one split and return per-image stats."""
    records = []
    img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    if not img_dir.exists():
        raise FileNotFoundError(f"Image directory not found: {img_dir}")

    for img_path in sorted(img_dir.iterdir()):
        if img_path.suffix.lower() not in img_exts:
            continue

        lbl_path = lbl_dir / (img_path.stem + ".txt")
        bboxes, classes = [], []

        if lbl_path.exists():
            text = lbl_path.read_text().strip()
            if text:
                for line in text.splitlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cls, cx, cy, w, h = parts
                        classes.append(int(cls))
                        bboxes.append((float(cx), float(cy), float(w), float(h)))

        records.append({
            "img_path": str(img_path),
            "lbl_path": str(lbl_path),
            "stem": img_path.stem,
            "n_obj": len(bboxes),
            "classes": classes,
            "bboxes": bboxes,
            "has_label": lbl_path.exists(),
        })

    return pd.DataFrame(records)

df_train = scan_split(TRAIN_IMG, TRAIN_LBL)
df_valid = scan_split(VALID_IMG, VALID_LBL)
df_test  = scan_split(TEST_IMG, TEST_LBL)

print(f"\n{'Split':<8} {'Images':>8} {'Labels':>8} {'Missing':>8} {'Total objs':>12}")
print("-" * 50)
for name, df in [("train", df_train), ("valid", df_valid), ("test", df_test)]:
    missing = (~df["has_label"]).sum()
    print(f"{name:<8} {len(df):>8} {df['has_label'].sum():>8} {missing:>8} {df['n_obj'].sum():>12}")

## 2. Sanity Check – Image ↔ Label Alignment

In [ ]:
def sanity_check(df: pd.DataFrame, split_name: str) -> bool:
    errors = []

    missing = df[~df["has_label"]]
    if len(missing):
        errors.append(f"⚠ {len(missing)} images without label files")

    empty = df[df["has_label"] & (df["n_obj"] == 0)]
    if len(empty):
        errors.append(f"⚠ {len(empty)} label files with 0 objects")

    bad_bbox = 0
    for _, row in df.iterrows():
        for (cx, cy, w, h) in row["bboxes"]:
            if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < w <= 1 and 0 < h <= 1):
                bad_bbox += 1
    if bad_bbox:
        errors.append(f"⚠ {bad_bbox} bboxes outside [0,1] range")

    all_cls = [c for row in df["classes"] for c in row]
    out_of_range = [c for c in all_cls if c < 0 or c >= len(CLASS_NAMES)]
    if out_of_range:
        errors.append(f"⚠ {len(out_of_range)} class indices out of range")

    status = "✅ PASS" if not errors else "❌ FAIL"
    print(f"\n[{split_name}] Sanity Check → {status}")
    for e in errors:
        print(" ", e)
    if not errors:
        print(f"  All {len(df)} images matched correctly.")
    return len(errors) == 0

all_ok = True
for name, df in [("train", df_train), ("valid", df_valid), ("test", df_test)]:
    all_ok &= sanity_check(df, name)

assert all_ok, "Sanity check failed – fix data before proceeding."

## 3. BBox Statistics (Small-Object Analysis)

In [ ]:
def compute_bbox_stats(df: pd.DataFrame) -> dict:
    px_w, px_h, areas = [], [], []
    for _, row in df.iterrows():
        for (cx, cy, w, h) in row["bboxes"]:
            pw = w * IMG_SIZE
            ph = h * IMG_SIZE
            px_w.append(pw)
            px_h.append(ph)
            areas.append((pw * ph) / (IMG_SIZE ** 2))

    stats = {
        "total_objects": len(px_w),
        "mean_w_px": float(np.mean(px_w)) if px_w else 0.0,
        "mean_h_px": float(np.mean(px_h)) if px_h else 0.0,
        "mean_area_pct": float(np.mean(areas) * 100) if areas else 0.0,
        "median_area_pct": float(np.median(areas) * 100) if areas else 0.0,
        "mean_objects_img": float(df["n_obj"].mean()) if len(df) else 0.0,
        "max_objects_img": int(df["n_obj"].max()) if len(df) else 0,
        "small_obj_pct": float(np.mean([a < 0.01 for a in areas]) * 100) if areas else 0.0,
    }
    return stats

stats_train = compute_bbox_stats(df_train)
print("\n📊 BBox Statistics (Train):")
for k, v in stats_train.items():
    print(f"  {k:<25}: {v:.3f}" if isinstance(v, float) else f"  {k:<25}: {v}")

print(f"\n  Expected avg W≈36px, H≈49px, area≈0.45%")
print(f"  Got      avg W={stats_train['mean_w_px']:.1f}px, H={stats_train['mean_h_px']:.1f}px, area={stats_train['mean_area_pct']:.2f}%")

## 4. Class Distribution

In [ ]:
def class_distribution(df: pd.DataFrame, split: str):
    cnt = Counter(c for row in df["classes"] for c in row)
    total = sum(cnt.values()) or 1
    print(f"\n[{split}] Class distribution:")
    max_count = max(cnt.values()) if cnt else 1
    for i, name in enumerate(CLASS_NAMES):
        n = cnt.get(i, 0)
        bar = "█" * int(n / max_count * 30) if max_count else ""
        print(f" {name:<3}: {n:>5} ({n/total*100:5.1f}%) {bar}")
    counts = [cnt.get(i, 0) for i in range(len(CLASS_NAMES))]
    ratio = max(counts) / max(min([c for c in counts if c > 0], default=1), 1)
    print(f"  Imbalance ratio (max/min): {ratio:.2f}×")
    return {str(i): cnt.get(i, 0) for i in range(len(CLASS_NAMES))}

dist_train = class_distribution(df_train, "train")
dist_valid = class_distribution(df_valid, "valid")
dist_test  = class_distribution(df_test, "test")


## 5. Create 5% Labeled Split (Stratified)

In [ ]:
def create_stratified_split(df: pd.DataFrame, ratio: float, seed: int):
    """Stratified split by dominant class per image. Falls back to random split."""
    def dominant_class(classes):
        if not classes:
            return -1
        return Counter(classes).most_common(1)[0][0]

    df = df.copy()
    df["dom_class"] = df["classes"].apply(dominant_class)
    df_valid_for_split = df[df["n_obj"] > 0].copy()

    try:
        sss = StratifiedShuffleSplit(
            n_splits=1,
            train_size=1.0 - ratio,
            random_state=seed
        )
        unlabeled_idx, labeled_idx = next(
            sss.split(df_valid_for_split, df_valid_for_split["dom_class"])
        )
        df_labeled = df_valid_for_split.iloc[labeled_idx].copy()
        df_unlabeled = df_valid_for_split.iloc[unlabeled_idx].copy()
        print("✅ Stratified split succeeded")
    except Exception as e:
        print(f"  ⚠ Stratified split failed ({e}), using random split")
        n_labeled = max(1, int(len(df_valid_for_split) * ratio))
        rng = np.random.RandomState(seed)
        idx = rng.permutation(len(df_valid_for_split))
        df_labeled = df_valid_for_split.iloc[idx[:n_labeled]].copy()
        df_unlabeled = df_valid_for_split.iloc[idx[n_labeled:]].copy()

    return df_labeled, df_unlabeled

df_labeled, df_unlabeled = create_stratified_split(df_train, LABELED_RATIO, SEED)

print(f"\n5% Split Results:")
print(f"  Labeled   : {len(df_labeled):>5} images ({len(df_labeled)/len(df_train)*100:.2f}%)")
print(f"  Unlabeled : {len(df_unlabeled):>5} images ({len(df_unlabeled)/len(df_train)*100:.2f}%)")
print(f"  Total     : {len(df_labeled)+len(df_unlabeled):>5} (orig: {len(df_train)})")

print(f"\n  Labeled subset class distribution:")
lab_cnt = Counter(c for row in df_labeled["classes"] for c in row)
unlab_cnt = Counter(c for row in df_unlabeled["classes"] for c in row)
for i, name in enumerate(CLASS_NAMES):
    l_pct = lab_cnt.get(i, 0) / max(sum(lab_cnt.values()), 1) * 100
    u_pct = unlab_cnt.get(i, 0) / max(sum(unlab_cnt.values()), 1) * 100
    print(f"    {name:<8}: labeled={l_pct:.1f}% | unlabeled={u_pct:.1f}%")

## 6. Save Split Artifact

In [ ]:
split_artifact = {
    "seed": SEED,
    "labeled_ratio": LABELED_RATIO,
    "n_labeled": len(df_labeled),
    "n_unlabeled": len(df_unlabeled),
    "n_valid": len(df_valid),
    "n_test": len(df_test),
    "labeled_stems": sorted(df_labeled["stem"].tolist()),
    "unlabeled_stems": sorted(df_unlabeled["stem"].tolist()),
    "valid_stems": sorted(df_valid["stem"].tolist()),
    "test_stems": sorted(df_test["stem"].tolist()),
    "class_names": CLASS_NAMES,
    "nc": NC,
    "img_size": IMG_SIZE,
}

split_path = OUT_DIR / "split_5pct.json"
with open(split_path, "w") as f:
    json.dump(split_artifact, f, indent=2, ensure_ascii=False)

print(f"✅ Split saved to: {split_path}")

with open(split_path, "r") as f:
    loaded = json.load(f)
assert loaded["n_labeled"] == len(df_labeled)
assert loaded["seed"] == SEED
print(f"    Reload check  : n_labeled={loaded['n_labeled']} ✓  seed={loaded['seed']} ✓")

stats_artifact = {
    "bbox_stats_train": stats_train,
    "class_dist_train": dist_train,
    "class_dist_valid": dist_valid,
    "class_dist_test": dist_test,
}
stats_path = OUT_DIR / "dataset_stats.json"
with open(stats_path, "w") as f:
    json.dump(stats_artifact, f, indent=2, ensure_ascii=False)
print(f"    Stats saved to: {stats_path}")

## 7. Visualization – Sample Images + BBoxes

In [ ]:
# ── Sample Visualization with Class Legend ────────────────────

COLORS = [
    "#e6194B",  # red
    "#3cb44b",  # green
    "#4363d8",  # blue
    "#f58231",  # orange
    "#911eb4",  # purple
]

# ── Draw image + bbox ─────────────────────────────────────────
def draw_sample(img_path: str, lbl_path: str, ax, title: str = ""):
    img = Image.open(img_path).convert("RGB")
    W, H = img.size

    ax.imshow(img)
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.axis("off")

    if not Path(lbl_path).exists():
        return

    txt = Path(lbl_path).read_text().strip()
    if not txt:
        return

    for line in txt.splitlines():

        parts = line.strip().split()
        if len(parts) != 5:
            continue

        cls = int(parts[0])
        cx, cy, w, h = map(float, parts[1:])

        x1 = (cx - w / 2) * W
        y1 = (cy - h / 2) * H

        color = COLORS[cls % len(COLORS)]

        # Bounding box
        rect = patches.Rectangle(
            (x1, y1),
            w * W,
            h * H,
            linewidth=2,
            edgecolor=color,
            facecolor="none"
        )
        ax.add_patch(rect)

        # Label text
        ax.text(
            x1,
            y1 - 3,
            CLASS_NAMES[cls],
            fontsize=6,
            color="white",
            fontweight="bold",
            bbox=dict(
                boxstyle="round,pad=0.2",
                fc=color,
                ec="none",
                alpha=0.9
            )
        )

# ── Sample groups ─────────────────────────────────────────────
sample_groups = [
    (df_labeled.sample(min(2, len(df_labeled)), random_state=SEED), "Labeled"),
    (df_unlabeled.sample(min(2, len(df_unlabeled)), random_state=SEED), "Unlabeled"),
    (df_valid.sample(min(2, len(df_valid)), random_state=SEED), "Valid"),
    (df_test.sample(min(2, len(df_test)), random_state=SEED), "Test"),
]

# ── Create figure ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

fig.suptitle(
    "PCB Dataset – Sample Images with Bounding Boxes",
    fontsize=15,
    fontweight="bold",
    y=1.02
)

# ── Draw samples ──────────────────────────────────────────────
for col, (grp, label) in enumerate(sample_groups):

    for row in range(2):

        ax = axes[row][col]

        if row < len(grp):

            rec = grp.iloc[row]

            draw_sample(
                rec["img_path"],
                rec["lbl_path"],
                ax,
                title=f"[{label}] {rec['stem']} | Objects: {rec['n_obj']}"
            )

        else:
            ax.axis("off")

# ── Legend ────────────────────────────────────────────────────
legend_handles = [
    patches.Patch(
        color=COLORS[i],
        label=f"{CLASS_NAMES[i]}"
    )
    for i in range(len(CLASS_NAMES))
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=len(CLASS_NAMES),
    fontsize=10,
    frameon=True,
    bbox_to_anchor=(0.5, -0.02)
)

# ── Layout & save ─────────────────────────────────────────────
plt.tight_layout()

viz_path = OUT_DIR / "sample_images.png"

fig.savefig(
    viz_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print(f"✅ Visualization saved: {viz_path}")

## 8. BBox Size Distribution Plot

In [ ]:
# ── Clean & Professional BBox Visualization ───────────────────

plt.style.use("seaborn-v0_8-whitegrid")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Prepare data ──────────────────────────────────────────────
all_bboxes = [
    (w * IMG_SIZE, h * IMG_SIZE)
    for _, row in df_train.iterrows()
    for (_, _, w, h) in row["bboxes"]
]

if all_bboxes:
    ws, hs = map(np.array, zip(*all_bboxes))
else:
    ws, hs = np.array([]), np.array([])

areas_pct = (ws * hs) / (IMG_SIZE ** 2) * 100 if len(ws) else np.array([])

# ── Plot helper ───────────────────────────────────────────────
def draw_hist(ax, data, title, xlabel, hist_color):
    ax.hist(
        data,
        bins=50,
        color=hist_color,
        alpha=0.85,
        edgecolor="black",
        linewidth=0.4
    )

    if len(data):
        mean_val = np.mean(data)
        median_val = np.median(data)

        # Mean line
        ax.axvline(
            mean_val,
            color="red",
            linestyle="--",
            linewidth=2,
            label=f"Mean: {mean_val:.2f}"
        )

        # Median line
        ax.axvline(
            median_val,
            color="blue",
            linestyle=":",
            linewidth=2,
            label=f"Median: {median_val:.2f}"
        )

    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel("Frequency", fontsize=11)

    ax.legend(fontsize=9, frameon=True)

    # Clean border
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# ── Draw plots ────────────────────────────────────────────────
draw_hist(
    axes[0],
    ws,
    "BBox Width Distribution",
    "Width (pixels)",
    "#4C72B0"
)

draw_hist(
    axes[1],
    hs,
    "BBox Height Distribution",
    "Height (pixels)",
    "#55A868"
)

draw_hist(
    axes[2],
    areas_pct,
    "BBox Area Distribution",
    "Area (% of image)",
    "#C44E52"
)

# ── Global title ──────────────────────────────────────────────
small_obj_pct = np.mean(areas_pct < 1.0) * 100 if len(areas_pct) else 0

fig.suptitle(
    f"PCB Dataset Bounding Box Statistics | Small Objects (<1%): {small_obj_pct:.1f}%",
    fontsize=16,
    fontweight="bold",
    y=1.02
)

# ── Save ──────────────────────────────────────────────────────
plt.tight_layout()

plot_path = OUT_DIR / "bbox_stats.png"

fig.savefig(
    plot_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print(f"✅ Plot saved: {plot_path}")

## 9. Final Artifact Checklist

In [ ]:
print("\n" + "="*55)
print("Notebook 1 – Final Artifact Checklist")
print("="*55)

required = {
    "split_5pct.json": "5% labeled split (reproducible)",
    "dataset_stats.json": "BBox and class statistics",
    "sample_images.png": "Visual samples from all splits",
    "bbox_stats.png": "BBox size distribution",
}

all_saved = True
for fname, desc in required.items():
    path = OUT_DIR / fname
    exists = path.exists()
    size = f"{path.stat().st_size // 1024}KB" if exists else "—"
    status = "✅" if exists else "❌"
    print(f"  {status} {fname:<25} ({size:<8}) – {desc}")
    all_saved &= exists

print("\n" + ("✅ All artifacts saved correctly!" if all_saved else "❌ Some artifacts missing – check above"))

print("\nSummary:")
print(f"  Train total    : {len(df_train)} images")
print(f"  5% Labeled     : {len(df_labeled)} images")
print(f"  Unlabeled      : {len(df_unlabeled)} images")
print(f"  Valid          : {len(df_valid)} images")
print(f"  Test           : {len(df_test)} images")
print(f"  Avg obj/image  : {stats_train['mean_objects_img']:.1f}")
print(f"  Avg bbox area  : {stats_train['mean_area_pct']:.3f}%")
print("\nNext: Run Notebook 2 (Baseline: Deformable DETR Supervised)")